# Train a YOLOv8 face-mask detector (Google Colab)

1. **Runtime → Change runtime type → T4 GPU**
2. Run each cell from top to bottom. Training takes about 20-30 minutes.
3. At the end, download `best.pt` and the result charts, and add them to the GitHub repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics kaggle

## 1. Get the code and the data

Change `GITHUB_USER` to your GitHub username.

For the data you need a Kaggle API key: kaggle.com → Settings → API → **Create New Token**. This downloads `kaggle.json`; upload it when the cell asks.

In [ ]:
GITHUB_USER = "Lavanyasinghhh"
!git clone https://github.com/{GITHUB_USER}/face-mask-detection.git
%cd face-mask-detection

In [ ]:
from google.colab import files
files.upload()  # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d andrewmvd/face-mask-detection -p data/raw --unzip
!ls data/raw && ls data/raw/images | wc -l

## 2. Convert annotations to YOLO format and split 70 / 15 / 15

In [ ]:
!python -m src.voc_to_yolo --src data/raw --out data/yolo

In [ ]:
# Check class balance: is any class rare?
from collections import Counter
from pathlib import Path

names = ["with_mask", "without_mask", "mask_weared_incorrect"]
for split in ["train", "val", "test"]:
    c = Counter()
    for f in Path(f"data/yolo/labels/{split}").glob("*.txt"):
        c.update(names[int(l.split()[0])] for l in f.read_text().splitlines() if l)
    print(split, dict(c))

`mask_weared_incorrect` is much rarer than the other two classes, so expect lower accuracy on it. Mention this in the README.

## 3. Train

Starting from YOLOv8n weights pre-trained on COCO (transfer learning). `patience=15` stops early if validation mAP stops improving.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="data/yolo/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    seed=42,
    project="runs",
    name="mask",
)

## 4. Evaluate on the held-out test set

The model never saw these images during training or model selection.

In [ ]:
best = YOLO("runs/mask/weights/best.pt")
metrics = best.val(data="data/yolo/data.yaml", split="test", project="runs", name="test")

print(f"mAP@0.5      : {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.3f}")
print(f"Precision    : {metrics.box.mp:.3f}")
print(f"Recall       : {metrics.box.mr:.3f}")
for i, name in enumerate(names):
    print(f"  {name:24s} AP@0.5 = {metrics.box.ap50[i]:.3f}")

In [ ]:
from IPython.display import Image, display
display(Image("runs/mask/results.png", width=900))
display(Image("runs/test/confusion_matrix_normalized.png", width=600))

## 5. Look at some predictions

In [ ]:
import random
test_imgs = list(Path("data/yolo/images/test").glob("*"))
for img in random.sample(test_imgs, 4):
    r = best.predict(str(img), conf=0.35, verbose=False)[0]
    r.save(filename=f"pred_{img.stem}.jpg")
    display(Image(f"pred_{img.stem}.jpg", width=500))

## 6. Download the weights and charts for the repo

Put `best.pt` in `models/`. Put the PNGs in `docs/` and update the results table in the README with the numbers printed in step 4.

In [ ]:
import shutil
!mkdir -p export
shutil.copy("runs/mask/weights/best.pt", "export/best.pt")
shutil.copy("runs/mask/results.png", "export/training_curves.png")
shutil.copy("runs/test/confusion_matrix_normalized.png", "export/confusion_matrix.png")
for p in Path(".").glob("pred_*.jpg"):
    shutil.copy(p, "export/" + p.name)
!zip -r export.zip export
files.download("export.zip")